# Denoising Objective Comparison: Predict x0 vs. Predict Noise

Two training objectives for the exact same architecture (`TinyUNet64`,
`code/unet_tiny_denoising_64.py`) on the exact same task (CIFAR-10
resized to 64x64, additive Gaussian noise, same `noise_std`):

| objective | network output | loss | final activation |
|---|---|---|---|
| `"x0"` | `x_hat` directly | `L = ||x - x_hat||^2` | `Sigmoid` (output in `[0,1]`) |
| `"noise"` | `eps_hat` | `L = ||eps - eps_hat||^2` | identity (unbounded, zero-mean) |

where `x_noisy = x + sigma*z`, `z ~ N(0,I)`, `sigma = noise_std`, and
`eps := x_noisy - x` is the actual (already-clamped) corruption applied
to that image. For the `"noise"` objective the clean-image estimate is
recovered afterward via `x_hat = clamp(x_noisy - eps_hat, 0, 1)`.

This mirrors the two equivalent parameterizations of a DDPM
noise-predictor U-Net (predict `x0` vs. predict `eps`; Ho et al., 2020,
Sec. 3.2) -- architecture and available information are identical
between the two objectives, so any gap in denoising quality isolates
the effect of *what the loss is computed against*.

**Important**: the two objectives' raw training losses are in
different units (pixel-space MSE vs. noise-space MSE) and are **not**
directly comparable as numbers. The fair, common metric used
throughout this notebook is the MSE of the recovered clean-image
estimate `x_hat` against the true clean image (`test_denoising_mse`),
which is in identical units for both objectives.

Core code lives in `code/unet_tiny_denoising_64.py` (the `TinyUNet64`
model, both training objectives, and `denoise_to_clean_image`) and
`code/compare_denoising_objectives.py` (the comparison/plotting logic),
both version-controlled; this notebook is a thin orchestration layer:
GPU check, repo pull, build model + print the full shape trace (shared
by both objectives -- the architecture does not depend on
`prediction_mode`), download real CIFAR-10 resized to 64x64 (cached in
Drive), train both objectives back-to-back with identical
hyperparameters, plot native-loss curves plus the comparable
denoising-MSE curve, and show clean/noisy/`x0`-denoised/`noise`-denoised
qualitative + quantitative results as the required
results-visualization step.

**What is controlled vs. not**: random seed (reset immediately before
each model's construction, so both start from bit-identical initial
weights), epochs, batch size, learning rate, optimizer, `noise_std`,
and the dataset/split are identical for both objectives. The exact
Gaussian-noise draws and mini-batch order during each objective's own
training loop are not bit-identical between the two runs (same caveat
as `compare_stl10_autoencoder_vs_unet.ipynb`; see CLAUDE.md Research
Integrity).

## 1. GPU check

In [ ]:
import torch

# Never hard-code CUDA: fall back to CPU if no GPU is available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


## 2. Clone/pull repository

Model code lives in `code/unet_tiny_denoising_64.py`, comparison code in
`code/compare_denoising_objectives.py`, not duplicated in this notebook.
Replace `REPO_URL` with this repository's URL.

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code"))


## 3. Build model, print full shape trace

One verbose forward pass on a small synthetic batch, purely to print
the `[B,C,H,W]` shape before and after every op -- including every
skip-connection save and every `torch.cat`. The architecture (and
therefore this trace) does not depend on `prediction_mode`, so this is
printed once using the default `"x0"` mode; not part of training.

In [ ]:
from unet_tiny_denoising_64 import TinyUNet64, count_trainable_parameters

torch.manual_seed(0)

BASE_CHANNELS = 32

shape_trace_model = TinyUNet64(in_channels=3, out_channels=3, base_channels=BASE_CHANNELS).to(device)
print(f"trainable parameters: {count_trainable_parameters(shape_trace_model):,}")

sample_batch = torch.rand(4, 3, 64, 64, device=device)  # [B,3,64,64], synthetic, in [0,1]
_ = shape_trace_model(sample_batch, verbose=True)


## 4. Download real CIFAR-10 (train/test split), resized to 64x64, cached in Drive

CIFAR-10 is natively 32x32; `get_cifar10_denoising_dataloaders` resizes
every image up to 64x64 so it matches the resolution `TinyUNet64` was
designed for. Labels are unused (the denoising target is the clean
image itself) -- `torchvision`'s native train/test partition, no extra
validation split invented. Both objectives reuse the exact same two
DataLoaders.

Colab's local disk (`/content/...`) is wiped whenever the runtime
recycles, so we mount Google Drive and point `DATA_ROOT` at a
persistent Drive folder: `torchvision` downloads into it once and
reads from it on every later run.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Persistent cache: downloaded once into Drive, reused on every
# future Colab session (local /content storage is ephemeral).
DATA_ROOT = "/content/drive/MyDrive/thesis_data/cifar10"


## 5. Train both objectives with identical hyperparameters

`run_comparison` builds the train/test DataLoaders once (shared by both
objectives), then trains a `"x0"` and a `"noise"` `TinyUNet64` from
scratch, back-to-back, same seed/epochs/batch size/learning
rate/`noise_std`, printing one line per epoch per objective
(`[x0   ] epoch ...` / `[noise] epoch ...`) with both the native loss
and the common `test_denoising_mse` metric.

In [ ]:
from compare_denoising_objectives import run_comparison, print_comparison_summary

BATCH_SIZE = 128
N_EPOCHS = 10
LR = 1e-3
NOISE_STD = 0.3  # std of additive Gaussian noise defining the denoising task
SEED = 0

results = run_comparison(
    DATA_ROOT,
    batch_size=BATCH_SIZE,
    epochs=N_EPOCHS,
    lr=LR,
    noise_std=NOISE_STD,
    base_channels=BASE_CHANNELS,
    device=device,
    seed=SEED,
)

print_comparison_summary(results["summary"])


## 6. Loss curves: native (own units, not comparable) vs. test denoising MSE (comparable)

Left panel: each objective's own train/test loss, in its own units --
useful to confirm both are converging, but the two curves are not on a
shared scale. Right panel: `test_denoising_mse` per epoch for both
objectives on the same axes -- this is the actual, apples-to-apples
comparison of the two training objectives.

In [ ]:
from compare_denoising_objectives import plot_loss_comparison

plot_loss_comparison(results["histories"])


## 7. Qualitative + quantitative check: clean vs. noisy vs. both objectives' denoised output

Per-project convention: always look at concrete examples, not only
scalar metrics. 4 rows for 5 real test images, all sharing the same
noise draw:
1. clean (ground truth)
2. noisy input
3. `"x0"`-objective reconstruction (MSE/PSNR vs. clean)
4. `"noise"`-objective reconstruction (MSE/PSNR vs. clean)

In [ ]:
from compare_denoising_objectives import visualize_denoising_comparison

sample_images, _sample_labels = next(iter(results["test_loader"]))
sample_images = sample_images.to(device)

visualize_denoising_comparison(results["models"], sample_images, results["noise_std"], num_examples=5)


## 8. Final summary table

Parameter count (identical for both -- same architecture), final
test-set native loss (own units, not comparable across rows), and
final test-set denoising MSE/PSNR (comparable), repeated here for
convenience after training.

In [ ]:
print_comparison_summary(results["summary"])
